# La validación y la métrica de GrantMatch

Valido con validación cruzada el modelo final de `experimentos_proyecto.ipynb` (la corrida con `candidato` = `final`) y el segundo mejor, con dos esquemas:

- **Por grupo (`GroupKFold` por `award_key`), el que corresponde a mis datos:** el mismo award aparece en muchos pares (todas sus publicaciones financiadas, más sus pares negativos). Si un award cae a la vez en entrenamiento y en validación, el modelo se evalúa con convocatorias que ya vio. Por grupo, cada award queda entero de un solo lado, que es lo que pasa en producción con una convocatoria nueva.
- **Al azar (`KFold` mezclado), para comparar.**

Después, la métrica del negocio: cuánto cuesta cada tipo de error. La tabla queda en `validacion_proyecto.csv`.

In [1]:
import mlflow
import numpy as np
import pandas as pd
from sklearn.dummy import DummyClassifier
from sklearn.ensemble import GradientBoostingClassifier, RandomForestClassifier
from sklearn.linear_model import LogisticRegression
from sklearn.metrics import confusion_matrix, recall_score
from sklearn.model_selection import GroupKFold, KFold, cross_val_predict, cross_validate
from sklearn.neural_network import MLPClassifier
from sklearn.pipeline import Pipeline
from pipeline_proyecto import CATEGORICAS, NUMERICAS, OBJETIVO, YA_LISTAS, cargar_datos, construir_preparacion

datos = cargar_datos()
X = datos[NUMERICAS + CATEGORICAS + YA_LISTAS]
y = datos[OBJETIVO]
grupos = datos["award_key"]
print("Pares:", len(datos), "  awards distintos:", grupos.nunique())

Pares: 16440   awards distintos: 7795


In [2]:
MODELOS = {
    "linea base": lambda: DummyClassifier(strategy="prior"),
    "logistica C=1": lambda: LogisticRegression(C=1.0, max_iter=1000),
    "logistica C=0.01": lambda: LogisticRegression(C=0.01, max_iter=1000),
    "bosque sin limite": lambda: RandomForestClassifier(n_estimators=200, random_state=42, n_jobs=-1),
    "bosque max_depth=8": lambda: RandomForestClassifier(n_estimators=200, max_depth=8, random_state=42, n_jobs=-1),
    "gradient boosting": lambda: GradientBoostingClassifier(random_state=42),
    "red neuronal": lambda: MLPClassifier(hidden_layer_sizes=(64,), max_iter=1000, random_state=42),
}

# El final y el segundo mejor, segun el registro de la sesion pasada.
experimento = mlflow.get_experiment_by_name("grantmatch")
corridas = mlflow.search_runs(experiment_ids=[experimento.experiment_id], order_by=["metrics.auc_prueba DESC"])
corridas = corridas[corridas["tags.mlflow.runName"] != "linea base"].drop_duplicates("tags.mlflow.runName")
final = corridas.loc[corridas["tags.candidato"] == "final", "tags.mlflow.runName"].iloc[0]
segundo = corridas.loc[corridas["tags.mlflow.runName"] != final, "tags.mlflow.runName"].iloc[0]
print("Modelo final:", final, "  segundo mejor:", segundo)

Modelo final: gradient boosting   segundo mejor: bosque max_depth=8


## Dos esquemas de validación cruzada

In [3]:
resultados = {}


def validar(nombre_modelo, nombre_esquema, esquema, con_grupos):
    pipeline = Pipeline([("preparar", construir_preparacion()), ("modelo", MODELOS[nombre_modelo]())])
    puntajes = cross_validate(pipeline, X, y, cv=esquema, groups=grupos if con_grupos else None, scoring="roc_auc")
    auc = puntajes["test_score"]
    clave = f"{nombre_esquema}, {nombre_modelo}"
    resultados[clave] = {
        "esquema": nombre_esquema, "modelo": nombre_modelo,
        "auc_promedio": round(auc.mean(), 4), "desviacion": round(auc.std(), 4),
        "peor_pliegue": round(auc.min(), 4), "pliegues": [round(a, 4) for a in auc],
    }
    print(f"{clave:40s} AUC promedio {auc.mean():.4f}  desviacion {auc.std():.4f}  pliegues {resultados[clave]['pliegues']}")


por_award = GroupKFold(n_splits=5)
al_azar = KFold(n_splits=5, shuffle=True, random_state=42)
for nombre in [final, segundo]:
    validar(nombre, "por award", por_award, con_grupos=True)
    validar(nombre, "al azar", al_azar, con_grupos=False)

tabla = pd.DataFrame(resultados.values())
tabla.to_csv("validacion_proyecto.csv", index=False)
tabla

por award, gradient boosting             AUC promedio 0.8972  desviacion 0.0048  pliegues [0.8998, 0.889, 0.8997, 0.9027, 0.8947]


al azar, gradient boosting               AUC promedio 0.8968  desviacion 0.0042  pliegues [0.8943, 0.8943, 0.8961, 0.905, 0.8941]


por award, bosque max_depth=8            AUC promedio 0.8934  desviacion 0.0043  pliegues [0.8957, 0.8864, 0.8965, 0.898, 0.8904]


al azar, bosque max_depth=8              AUC promedio 0.8938  desviacion 0.0033  pliegues [0.8902, 0.8938, 0.8921, 0.9, 0.8929]


,esquema,modelo,auc_promedio,desviacion,peor_pliegue,pliegues
0,por award,gradient boosting,0.8972,0.0048,0.8890,"[0.8998, 0.889, 0.8997, 0.9027, 0.8947]"
1,al azar,gradient boosting,0.8968,0.0042,0.8941,"[0.8943, 0.8943, 0.8961, 0.905, 0.8941]"
2,por award,bosque max_depth=8,0.8934,0.0043,0.8864,"[0.8957, 0.8864, 0.8965, 0.898, 0.8904]"
3,al azar,bosque max_depth=8,0.8938,0.0033,0.8902,"[0.8902, 0.8938, 0.8921, 0.9, 0.8929]"


In [4]:
diferencia = resultados[f"por award, {final}"]["auc_promedio"] - resultados[f"por award, {segundo}"]["auc_promedio"]
variacion = max(resultados[f"por award, {final}"]["desviacion"], resultados[f"por award, {segundo}"]["desviacion"])
print(f"Diferencia entre {final} y {segundo} (por award): {diferencia:+.4f}")
print(f"Variacion entre pliegues (la mayor desviacion): {variacion:.4f}")
print("La diferencia es mayor que la variacion entre pliegues" if abs(diferencia) > variacion
      else "La diferencia NO es mayor que la variacion entre pliegues: con estos datos no se distinguen")

Diferencia entre gradient boosting y bosque max_depth=8 (por award): +0.0038
Variacion entre pliegues (la mayor desviacion): 0.0048
La diferencia NO es mayor que la variacion entre pliegues: con estos datos no se distinguen


## La métrica del negocio

En GrantMatch, el error caro es el **falso negativo**: no recomendarle a un investigador un award que sí era para él. Puede perder un financiamiento de decenas de miles de dólares. Un **falso positivo** (recomendar un award que no le corresponde) le cuesta solo el tiempo de leer la convocatoria y descartarla, unos minutos.

Lo aproximo como **10 a 1**: perder un award pertinente cuesta 10 unidades y una recomendación que no sirve cuesta 1. Con esos costos busco el umbral de decisión de menor costo, y reporto el recall (qué fracción de los awards pertinentes llega a recomendarse), que la AUC no dice.

In [5]:
COSTO_FALSO_NEGATIVO = 10   # un award pertinente que no se recomienda
COSTO_FALSO_POSITIVO = 1    # una recomendacion que el investigador descarta
UMBRALES = [0.05, 0.1, 0.2, 0.3, 0.4, 0.5, 0.6, 0.7]  # con costos 10 a 1, el optimo teorico es 1/11 = 0.09

pipeline_final = Pipeline([("preparar", construir_preparacion()), ("modelo", MODELOS[final]())])
probabilidades = cross_val_predict(pipeline_final, X, y, cv=por_award, groups=grupos, method="predict_proba")[:, 1]

costos = {}
for umbral in UMBRALES:
    prediccion = (probabilidades >= umbral).astype(int)
    (vn, fp), (fn, vp) = confusion_matrix(y, prediccion)
    costos[umbral] = fn * COSTO_FALSO_NEGATIVO + fp * COSTO_FALSO_POSITIVO
    print(f"Umbral {umbral:.2f}: recall {recall_score(y, prediccion):.3f}   {fn:5d} pertinentes perdidos   {fp:5d} recomendaciones de mas   costo {costos[umbral]:7d}")

umbral_elegido = min(costos, key=costos.get)
print("Umbral de menor costo:", umbral_elegido, " (con 0.5 el costo seria", costos[0.5], ")")

Umbral 0.05: recall 0.999       7 pertinentes perdidos    8127 recomendaciones de mas   costo    8197
Umbral 0.10: recall 0.982     144 pertinentes perdidos    5891 recomendaciones de mas   costo    7331
Umbral 0.20: recall 0.919     668 pertinentes perdidos    2758 recomendaciones de mas   costo    9438
Umbral 0.30: recall 0.887     931 pertinentes perdidos    1970 recomendaciones de mas   costo   11280
Umbral 0.40: recall 0.871    1064 pertinentes perdidos    1673 recomendaciones de mas   costo   12313
Umbral 0.50: recall 0.853    1209 pertinentes perdidos    1429 recomendaciones de mas   costo   13519
Umbral 0.60: recall 0.816    1513 pertinentes perdidos    1142 recomendaciones de mas   costo   16272
Umbral 0.70: recall 0.755    2010 pertinentes perdidos     873 recomendaciones de mas   costo   20973
Umbral de menor costo: 0.1  (con 0.5 el costo seria 13519 )
